# DistressNet on Kaggle: train the voice model and run every evaluation

This is the run that produced the saved model and reports (`kaggle/distress-net-wavelm.ipynb`, cells 5–8), in one pass, from `main` of github.com/sarthak078/DIstress-Detection. It also retrains the Monitor CNN. Every step is one of the project's own scripts in `backend/app/`.

**Settings (right-hand panel, before running):**
1. **Session options → Accelerator → GPU T4 x2** (or P100).
2. **Session options → Internet → On.** It needs GitHub, pip and the WavLM weights from Hugging Face. Kaggle asks for a verified phone number the first time.
3. **Datasets:** nothing to download. The data cell attaches `uwrfkaggler/ravdess-emotional-speech-audio` and `ejlok1/cremad`; you can also add them yourself with **Add Input** and search for those names.

**Run:** **Save Version → Save & Run All (Commit)** runs it in the background (about 70 minutes on a T4, mostly the CNN folds). When it finishes, open the version's **Output** tab and download `distressnet_results.zip`. On the studio's Training page, choose **Load Kaggle results** and pick that zip: the new model goes live on Monitor and every report updates.

## Setup

In [ ]:
import os
import subprocess

REPO = "https://github.com/sarthak078/DIstress-Detection.git"
BRANCH = "main"
REPO_DIR = "/kaggle/working/DIstress-Detection"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "-q", "origin", BRANCH], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "log", "--oneline", "-1"], check=True)
# Kaggle's image already has torch, transformers and scikit-learn; opensmile is the one extra.
subprocess.run(["pip", "-q", "install", "opensmile"], check=True)

import torch

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none. Turn on a GPU under Session options, then run again.")

## Data: RAVDESS speech and CREMA-D from Kaggle datasets

In [ ]:
import glob
import shutil
from pathlib import Path


def find(pattern: str) -> list[str]:
    return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))


def attach(handle: str, pattern: str) -> list[str]:
    """The dataset when it is already attached; otherwise attach it with kagglehub (inside Kaggle it is mounted, not downloaded)."""
    hits = find(pattern)
    if not hits:
        import kagglehub

        root = kagglehub.dataset_download(handle)
        hits = sorted(glob.glob(f"{root}/**/{pattern}", recursive=True))
    assert hits, f"Could not find {handle}. Add it with Add Input, then run this cell again."
    return hits


# The RAVDESS dataset holds every actor twice (top level and audio_speech_actors_01-24/); use one copy.
ravdess = os.path.dirname(attach("uwrfkaggler/ravdess-emotional-speech-audio", "audio_speech_actors_01-24/Actor_01")[0])
crema = os.path.dirname(attach("ejlok1/cremad", "1001_DFA_ANG_XX.wav")[0])

data = Path(REPO_DIR) / "data"
data.mkdir(exist_ok=True)
for name, target in (("ravdess", ravdess), ("crema-d", crema)):
    link = data / name
    if link.is_symlink() or link.is_file():
        link.unlink()
    elif link.is_dir():
        shutil.rmtree(link)
    link.symlink_to(target)
print("RAVDESS:", ravdess, len(glob.glob(f"{data}/ravdess/Actor_*/03-01-*.wav")), "clips")
print("CREMA-D:", crema, len(glob.glob(f"{data}/crema-d/*.wav")), "clips")

# Keep the saved results for the comparison at the end, then start the reports from empty:
# the scripts merge into an existing report, and this run should stand on its own.
ARTIFACTS = Path(REPO_DIR) / "backend" / "artifacts"
REPORTS = ["cv_report.json", "cv_report_crema-d.json", "cross_corpus_report.json", "reader_report.json", "comparison.md"]
SAVED = Path("/kaggle/working/saved_results")
SAVED.mkdir(exist_ok=True)
for name in REPORTS + ["voice_head.pt", "voice_report.json", "audio_branch.pt", "training_report.json"]:
    if (ARTIFACTS / name).exists():
        shutil.move(ARTIFACTS / name, SAVED / name)
print("Saved results kept in", SAVED)


def run(*args: str) -> None:
    """Run one project script from backend/ and stop if it fails."""
    print("\n$ python -m", " ".join(args), flush=True)
    subprocess.run(["python", "-m", *args], cwd=Path(REPO_DIR) / "backend", check=True)

## 1. Speaker cross-validation

RAVDESS: six folds by speaker, 4 validation speakers each. CREMA-D: five folds, 9 validation speakers. Models: the log-mel CNN, eGeMAPS with logistic regression, and frozen WavLM-base-plus with the four-class attentive head, then the binary mean-pooled head. The first WavLM run embeds every clip once and caches it.

In [ ]:
run("app.cv")
run("app.cv", "--only", "wavlm", "--head", "binary-mean")
run("app.cv", "--corpus", "crema-d")
run("app.cv", "--corpus", "crema-d", "--only", "wavlm", "--head", "binary-mean")

## 2. Cross-corpus study

Speakers held out within RAVDESS and within CREMA-D, then train on one corpus and test on the other, both ways. Then with features standardized per corpus or per speaker (unlabeled audio), and the same for the binary-mean head.

In [ ]:
run("app.study")
for norm in ("corpus", "speaker"):
    run("app.study", "--models", "egemaps", "wavlm", "--norm", norm)
for norm in ("none", "corpus", "speaker"):
    run("app.study", "--models", "wavlm", "--head", "binary-mean", "--norm", norm)

## 3. Train the live voice model

The binary mean-pooled WavLM head. RAVDESS actors 01–18 plus all but nine CREMA-D speakers train; actors 19–20 and those nine speakers choose the epoch, temperature and threshold; actors 21–24 are scored once.

In [ ]:
run("app.voice")

## 3b. Retrain the Monitor CNN

The log-mel CNN on RAVDESS: actors 01–18 train, 19–20 choose the epoch, 21–24 are scored. This replaces the old `training_report.json`, which came from an earlier 01–06 / 07–08 run.

In [ ]:
run("app.train")


## 4. Words reader and comparison table

The phrase list against GoEmotions RoBERTa on 200 labeled sentences. The language-model reader needs Ollama or an OpenAI key, so it stays on the studio machine. Then one table of every model against the proposal targets.

In [ ]:
run("app.reader_eval", "--readers", "phrases", "roberta")
run("app.compare")
print((ARTIFACTS / "comparison.md").read_text())

## 5. Compare with the saved results and package the output

In [ ]:
import json


def head_summary(path: Path) -> dict:
    return torch.load(path, map_location="cpu", weights_only=True)["summary"]


if (SAVED / "voice_head.pt").exists():
    old, new = head_summary(SAVED / "voice_head.pt"), head_summary(ARTIFACTS / "voice_head.pt")
    print("Live voice model (actors 21–24)        saved   this run")
    for key in ("selected_epoch", "threshold", "test_balanced_accuracy", "test_auroc"):
        print(f"  {key:<36}{old.get(key)!s:>8}{new.get(key)!s:>10}")


def cv_means(path: Path) -> dict:
    report = json.loads(path.read_text())
    return {name: block["summary_calibrated"]["balanced_accuracy"]["mean"] for name, block in report["models"].items()}


for name in ("cv_report.json", "cv_report_crema-d.json"):
    if (SAVED / name).exists():
        saved, now = cv_means(SAVED / name), cv_means(ARTIFACTS / name)
        print(f"\n{name}: calibrated balanced accuracy, mean over folds")
        for model in saved:
            print(f"  {model:<36}{saved[model]:>8.3f}{now.get(model, float('nan')):>10.3f}")

if (SAVED / "cross_corpus_report.json").exists():
    saved, now = (json.loads((p / "cross_corpus_report.json").read_text())["models"] for p in (SAVED, ARTIFACTS))
    print("\ncross_corpus_report.json: calibrated balanced accuracy")
    for model, rows in saved.items():
        for i, row in enumerate(rows):
            again = now.get(model, [{}] * len(rows))[i].get("calibrated", {}).get("balanced_accuracy", float("nan"))
            print(f"  {model + ' / ' + row['name']:<62}{row['calibrated']['balanced_accuracy']:>7.3f}{again:>8.3f}")

# The studio's Load Kaggle results button reads this zip.
OUTPUT = ["voice_head.pt", "voice_report.json", "audio_branch.pt", "training_report.json"] + REPORTS
out_dir = Path("/kaggle/working/distressnet_results")
shutil.rmtree(out_dir, ignore_errors=True)
out_dir.mkdir()
for name in OUTPUT:
    if (ARTIFACTS / name).exists():
        shutil.copy2(ARTIFACTS / name, out_dir / name)
shutil.make_archive("/kaggle/working/distressnet_results", "zip", out_dir)
print("\n/kaggle/working/distressnet_results.zip:", sorted(p.name for p in out_dir.iterdir()))

from IPython.display import FileLink

FileLink("distressnet_results.zip")